## 하이브리드 리랭킹

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w7" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
from sqlalchemy import func, inspect, select
from app.db.session import get_engine, session_scope
from app.models.document import Chunk
from app.models.eval import EvalRun, EvalWeakType

SKIP = ('.git', '.pytest_cache', '.venv', '__pycache__', 'sandbox')

files = [p for p in ROOT.rglob('*') if p.is_file() and (not any((s in p.parts for s in SKIP)))]

migrations = sorted((ROOT / 'backend/app/db/migrations/versions').glob('*.py'))

table_names = inspect(get_engine()).get_table_names()

print(f'프로젝트 파일 : {len(files)}개   (.git · .pytest_cache · .venv · __pycache__ · sandbox 제외)')
print(f'chunks 컬럼   : {len(Chunk.__table__.columns)}        ← 어제와 같습니다. 오늘도 칸을 안 늘립니다')
print(f'DB 표         : {len(table_names)}       ← 어제 저녁에 9 에서 11 이 됐습니다')
print(f'마이그레이션  : {len(migrations)}건      ← 오늘은 0건이 늘 예정입니다')
if 'eval_runs' not in table_names:
    print('eval_runs        : 표가 없습니다 — 아래 복원 셀이 만듭니다')
else:
    with session_scope() as s:
        runs = s.scalar(select(func.count()).select_from(EvalRun))
        weak = s.scalar(select(func.count()).select_from(EvalWeakType))
    print(f'eval_runs        : {runs}행   ← 어제 「기본 검색」 한 줄')
    print(f'eval_weak_types  : {weak}행   ← 어제 저녁의 오답 유형 넷')


프로젝트 파일 : 150개   (.git · .pytest_cache · .venv · __pycache__ · sandbox 제외)
chunks 컬럼   : 7        ← 어제와 같습니다. 오늘도 칸을 안 늘립니다
DB 표         : 11       ← 어제 저녁에 9 에서 11 이 됐습니다
마이그레이션  : 6건      ← 오늘은 0건이 늘 예정입니다
eval_runs        : 1행   ← 어제 「기본 검색」 한 줄
eval_weak_types  : 4행   ← 어제 저녁의 오답 유형 넷


In [3]:
from sqlalchemy import select
from app.db.session import session_scope
from app.models.eval import EvalWeakType

with session_scope() as s:
    weak_rows = [(w.ord, w.label, w.count, w.accuracy, w.diagnosis) for w in s.scalars(select(EvalWeakType).order_by(EvalWeakType.ord))]
    
if not weak_rows:
    print('어제 남긴 오답 유형이 없습니다 — 지난시간에 테스트를 돌리지 않았거나 DB 를 새로 만든 것입니다.')
for (ord_, label, cnt, acc, diag) in weak_rows:
    print(f'[{ord_ + 1}] {label:<18}{cnt:>3}건   정답률 {acc:>5}   ← {diag}')

[1] 타 부서 문서 참조          4건   정답률  50.0   ← 보안등급 정상 차단 · 시험지 오류 2건
[2] 규정에 없는 사항           6건   정답률     —   ← 근거부족 정상
[3] 표 안의 조건 조합          6건   정답률  83.3   ← 표가 조항 단위로 잘림
[4] 단일 조항              10건   정답률  90.0   ← 정답 문서가 상위에 없음


In [4]:
import json

REPORTS = ROOT / 'evals' / 'reports'
saved = sorted(REPORTS.glob('eval_*.json'))
wrong = []

if not saved:
    print('어제 리포트가 없습니다 — 터미널에서 run_evals 명령을 --no-save 없이 한 번 더 치면 파일이 떨어집니다.')
else:
    latest = saved[-1]
    report = json.loads(latest.read_text(encoding='utf-8'))
    wrong = [r for r in report['runs'][0]['results'] if r['ok'] is False]
    print(f'어제 리포트 : {latest.relative_to(ROOT).as_posix()}')
    print(f'틀린 문항 {len(wrong)}개 중 셋만 봅니다')
    for r in wrong[:3]:
        print(f"  {r['id']}  {r['type']:<12} ← {r['why']}")


어제 리포트 : evals/reports/eval_20261012_112637.json
틀린 문항 6개 중 셋만 봅니다
  G07  단일 조항        ← 정답 문서가 상위에 없음
  G16  표 안의 조건 조합   ← 정답 문서가 상위에 없음
  G23  타 부서 문서 참조   ← 정답 문서가 상위에 없음


In [5]:
from sqlalchemy import func, select
from app.db.session import session_scope
from app.models.document import Chunk

WORDS = ['출장비', '여비', '숙박', '1박당', '1박 당']

with session_scope() as s:
    for word in WORDS:
        n = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.text.like(f'%{word}%')))
        print(f"LIKE '%{word}%'".ljust(16), '→', f'{n}건')


LIKE '%출장비%'     → 0건
LIKE '%여비%'      → 33건
LIKE '%숙박%'      → 27건
LIKE '%1박당%'     → 2건
LIKE '%1박 당%'    → 0건


In [6]:
from app.rag.retriever import expand_queries

for question in ["출장비 얼마까지 나오나요?", "프로젝트 착수 단계 산출물은?"]:
    queries = expand_queries(question, 3)
    print(f'"{question}"  → {len(queries)}갈래')
    for i, one in enumerate(queries, 1):
        print(f"    [{i}] {one}")

"출장비 얼마까지 나오나요?"  → 2갈래
    [1] 출장비 얼마까지 나오나요?
    [2] 출장비 얼마까지 나오나요? 여비 일비 숙박비 식비
"프로젝트 착수 단계 산출물은?"  → 1갈래
    [1] 프로젝트 착수 단계 산출물은?


In [7]:
import collections
import yaml

items = yaml.safe_load((ROOT / 'evals' / 'golden_set.yaml').read_text(encoding='utf-8'))['items']
dist = collections.Counter()
picked = {}

for item in items:
    queries = expand_queries(item['question'], 3)
    dist[len(queries)] += 1
    picked.setdefault(len(queries), (item, queries))

total = sum(dist.values())
average = sum((k * v for (k, v) in dist.items())) / total
print(f'갈래 수 분포 : {dict(sorted(dist.items()))}   총 {total}질의 · 평균 {average:.2f}개')

for n in (1, 3):
    if n not in picked:
        continue
    (item, queries) = picked[n]
    print(f'''{n}갈래 예 : {item['id']}  "{item['question']}"''')
    for one in queries:
        print(f'             └ "{one}"')
        

갈래 수 분포 : {1: 14, 2: 10, 3: 6}   총 30질의 · 평균 1.73개
1갈래 예 : G01  "4급(대리) 직원의 국내출장 일비는 하루에 얼마인가요?"
             └ "4급(대리) 직원의 국내출장 일비는 하루에 얼마인가요?"
3갈래 예 : G02  "4급 직원이 광역시로 출장 갈 때 숙박비 상한은 얼마인가요?"
             └ "4급 직원이 광역시로 출장 갈 때 숙박비 상한은 얼마인가요?"
             └ "4급 직원이 광역시로 출장 갈 때 숙박비 상한은 얼마인가요? 숙박비 호텔 1박"
             └ "4급 직원이 광역시로 출장 숙박비 상한은 얼마인가요?"


In [ ]:
import importlib
from app.integrations.factory import get_embedder
from app.rag import retriever, store
from app.core.config import get_settings

importlib.reload(retriever)


EVAL_USER = {'id': '2011-0003', 'name': '한서영', 'clearance': '대외비', 'dept_id': 'HR'}
QUESTION = '출장비 얼마까지 나오나요?'

with session_scope() as s:
    settings = get_settings()
    embedder = get_embedder()
    (before, seen) = (0, set())
    for one in retriever.expand_queries(QUESTION, settings.multi_query_n):
        rows = store.vector_search(s, embedder.embed_query(one), limit=settings.retrieval_top_k)
        before += len(rows)
        seen.update((row['chunk_id'] for row in rows))
    result = retriever.LiveRetriever().search(question=QUESTION, user=EVAL_USER, session=s)
print(f"[1단계] {result.steps[0]['name']}   ({result.steps[0]['time']})")
print(f'중복 제거 전 : {before}건  →  후 : {len(seen)}건')

for (i, one) in enumerate(result.sources, 1):
    print(f"  {i}  {one['doc_id']}  {one['version']}  {one['locator']}   score {one['score']:.2f}")


In [ ]:
from sqlalchemy import select
from app.db.session import session_scope
from app.models.document import Chunk
from app.rag import embedder, store

NEEDLES = ['100,000', 'KTX']

with session_scope() as s:
    for needle in NEEDLES:
        locs = s.execute(select(Chunk.locator).where(Chunk.text.like(f'%{needle}%')).order_by(Chunk.id)).scalars().all()
        print(f"[글자] LIKE '%{needle}%'".ljust(23) + f'→  {len(locs)}건')
        for loc in locs[:3]:
            print(f'         · {loc}')
    rows = store.vector_search(s, embedder.embed_query(NEEDLES[0]), limit=3)
print(f'[벡터] "{NEEDLES[0]}" 상위 3건')
for (rank, row) in enumerate(rows, 1):
    print(f"         {rank} {row['locator']}   score {row['score']:.3f}")


[글자] LIKE '%100,000%'  →  1건
         · 표2
[글자] LIKE '%KTX%'      →  3건
         · 제3장 제3절 제18조(철도)
         · 표4
         · 제3장 제10조(교통비)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[벡터] "100,000" 상위 3건
         1 표2   score 0.451
         2 표1   score 0.442
         3 표1   score 0.442


In [ ]:
import importlib

from app.rag import store

importlib.reload(store)

print("숙박비는 1박당 70,000원")
print(store._tokenize("숙박비는 1박당 70,000원"))

숙박비는 1박당 70,000원
['숙박비는', '1박당', '70', '000원']


In [ ]:
from app.db.session import session_scope
from app.rag import embedder, store

importlib.reload(store)

QUESTION = "1~2급 직원의 숙박비 상한은 얼마인가요?"

with session_scope() as s: 
    vec = store.vector_search(s, embedder.embed_query(QUESTION), limit=3)
    key = store.keyword_search(s, QUESTION, limit=3)

print("질문 : ", QUESTION)
print()
print(" [ 벡터 ]".ljust(40) +"[키워드 BM25]")
for rank, (v, k) in enumerate(zip(vec, key), 1):
    left = f"{rank} {v['locator']}  {v['score']:.3f}"
    print(f" {left:<40}  {rank} {k['locator']} {k['score']:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

질문 :  1~2급 직원의 숙박비 상한은 얼마인가요?

 [ 벡터 ]                                 [키워드 BM25]
 1 표2  0.714                               1 제3장 제11조(숙박비) · 별표1 1.000
 2 제3장 제2절 제14조(숙박비) · 별표1  0.691          2 표2 0.849
 3 표2  0.690                               3 표2 0.848


In [ ]:
# 벡터 score를 남겨야하는 이유를 알 수 있는 테스트, 순위가 아닌 점수로 합치면 안되는 이유 
import importlib

from app.db.session import session_scope
from app.rag import store

importlib.reload(store)

QUESTION = "해외 출장 시 비지니스석 탑승 기준이 어떻게 되나요?"  # 문서에 없는 질문 

with session_scope() as s: 
    top = store.keyword_search(s, QUESTION, limit=3)

for row in top:
    shared = set(store._tokenize(QUESTION)) & set(store._tokenize(row["text"]))
    print(f"{row['score']:.2f}  {row['doc_id']} {row['locator'][:20]:20}  겹친 낱말 {sorted(shared)}")

print("max로 합친 최고 점수 : ", max((r["score"] for r in top), default=0.0))


1.00  DOC-HR-014 제2장 제7조(출장 신청)        겹친 낱말 ['출장']
0.98  DOC-HR-014 표3                    겹친 낱말 ['출장']
0.97  DOC-HR-014 제2장 제6조(출장 신청)        겹친 낱말 ['출장']
max로 합친 최고 점수 :  1.0


In [ ]:
import importlib
from app.db.session import session_scope
from app.rag import retriever, store

importlib.reload(store)
importlib.reload(retriever)

QUESTION = '1~2급 직원의 숙박비 상한은 얼마인가요?'
USER = {'name': '김민준', 'dept': 'INFRA2', 'clearance': '일반'}
with session_scope() as s:
    found = retriever.LiveRetriever().search(question=QUESTION, user=USER, session=s)
for (i, step) in enumerate(found.steps, 1):
    print(f"[{i}단계] {step['name']}  ({step['time']})   state={step['state']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[1단계] 질의 재작성 · Multi-Query 2건 생성  (0.0s)   state=ok
[2단계] 벡터 + 키워드 하이브리드 검색 · 벡터 14건 · 키워드 12건 -> 후보 18  (0.7s)   state=ok
